**Results check across k**

In [ ]:
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'retina'
import scienceplots
plt.style.use(['science', 'nature'])

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
# import EM_grouplasso_multimodal
import importlib
import nest_asyncio
nest_asyncio.apply()
import MCMC_evaluation

In [ ]:
from scipy.stats import gaussian_kde
from joblib import Parallel, delayed
import os

num_cores = os.cpu_count()

def calculate_marginal_map(samples):

    kde = gaussian_kde(samples)
    sample_min = np.min(samples)
    sample_max = np.max(samples)

    range_buffer = (sample_max - sample_min) * 0.1

    if range_buffer == 0:
        range_buffer = 1e-6
    
    eval_range = np.linspace(sample_min - range_buffer, sample_max + range_buffer, 10000)
    
    if len(eval_range) == 0 or np.all(np.isnan(eval_range)):
        return np.nan
    density_values = kde(eval_range)
    
    if len(density_values) == 0 or np.all(np.isnan(density_values)):
        return np.nan
        
    map_estimate = eval_range[np.argmax(density_values)]
    return map_estimate

*MCMC results*

In [ ]:
importlib.reload(MCMC_evaluation)

true_k = 3
N = 500
r = 5
tr = 1
dat_dir = "./Simulation/Simulation_1"
dat_name = f"dat_K{true_k}N{N}r{r}tr{tr}.npz"
sim_dat = np.load(dat_dir + dat_name)
X = sim_dat["X"]

X = np.concatenate((X, np.ones((X.shape[0], 1))), axis=1)

Y = sim_dat["Y"]

result_dir = f"./Simulation_results/Simulation_1/MCMC_results/K{true_k}N{N}r{r}tr{tr}/"
loo_values = []
dic_values = []
waic_values = []
for k in range(2,7):
    result = az.from_netcdf(result_dir + f"dat_K{k}.nc")
    evaluator = MCMC_evaluation.evaluation(X, Y, k, result)
    loo, dic, waic = evaluator.index()
    loo_values.append(loo)
    dic_values.append(dic)
    waic_values.append(waic)

print(loo_values, dic_values)

In [ ]:
K_values = [i for i in range(2,7)]
plt.figure()
plt.plot(K_values, loo_values, marker='o')
plt.xlabel('K')
plt.ylabel('LOO')
# plt.ylabel('LOO')
plt.xticks(K_values)
plt.grid(True)
# plt.show()

plt.savefig(f'./Fig_MCMC/LOO_k{true_k}n{N}r{r}tr{tr}.pdf', format='pdf', bbox_inches='tight', transparent=True)


**Results check**

*MCMC*

In [ ]:
# 1. compare with the true values
K = 3
N = 500
r = 10
tr = 5

result_dir = f"./Simulation_results/Simulation_1/MCMC_results/K{K}N{N}r{r}tr{tr}/"
result = az.from_netcdf(result_dir + f"dat_K{K}.nc")


dat_dir = "./Simulation/Simulation_1"
sim_dat = np.load(dat_dir + f"dat_K{K}N{N}r{r}tr{tr}.npz")

mu_true = sim_dat["mu"]
print(mu_true)
indice_true = np.argsort(mu_true)
mu_true = mu_true[indice_true]
sigma_true = sim_dat["sigma"][indice_true]

mu_samples = result.posterior['mu_k'].values.reshape(4000, K)
mu_est = Parallel(n_jobs=num_cores)(
            delayed(calculate_marginal_map)(mu_samples[:, i]) for i in range(K)
        )
mu_est = np.array(mu_est)
# mu_est = np.sort(az.summary(result, var_names=["mu_k"])['mean'].values)
indice_est = np.argsort(mu_est)
mu_est = mu_est[indice_est]

sigma_samples = result.posterior['sigma_k'].values.reshape(4000, K)
sigma_est = Parallel(n_jobs=num_cores)(
            delayed(calculate_marginal_map)(sigma_samples[:, i]) for i in range(K)
        )
sigma_est = np.array(sigma_est)
sigma_est = sigma_est[indice_est]
# sigma_est = az.summary(result, var_names=["sigma_k"])['mean'].values[indice_true]

rmse_mu = np.sqrt(np.mean((mu_true - mu_est)**2))
rmse_sigma = np.sqrt(np.mean((sigma_true-sigma_est)**2))

rrmse_mu = np.sqrt(np.mean((mu_true - mu_est)**2))/np.mean(mu_true)
rrmse_sigma = np.sqrt(np.mean((sigma_true-sigma_est)**2))/np.mean(sigma_true)

print('RMSE:', rmse_mu, rmse_sigma)
print('RRMSE:', rrmse_mu, rrmse_sigma)
print(mu_true, sigma_true)
print(mu_est, sigma_est)
print(sim_dat["theta"],sum(sim_dat["w"]))

In [ ]:
# 2 get AUC
from sklearn.metrics import auc
def calculate_auc(select_scores, true_labels, num_thresholds=1000):

    min_lamb = min(select_scores)
    max_lamb = max(select_scores)

    thresholds = np.linspace(min_lamb, max_lamb, 1000)
    tpr_list = []
    fpr_list = []

    for t in thresholds:
        y_pred = (select_scores > t).astype(int)
        TP = np.sum((y_pred == 1) & (true_labels == 1))
        FP = np.sum((y_pred == 1) & (true_labels == 0))
        FN = np.sum((y_pred == 0) & (true_labels == 1))
        TN = np.sum((y_pred == 0) & (true_labels == 0))

        TPR = TP / (TP + FN + 1e-8)
        FPR = FP / (FP + TN + 1e-8)

        tpr_list.append(TPR)
        fpr_list.append(FPR)

    # AUC
    manual_auc = auc(fpr_list, tpr_list)

    youden_j = np.array(tpr_list) - np.array(fpr_list)
    opt_index = np.argmax(youden_j)
    opt = thresholds[opt_index]
    # opt_tpr = tpr_list[opt_index]
    # opt_fpr = fpr_list[opt_index]

    # print(f"\nOptimal threshold (Youden's J): {opt:.4f}")
    # print(f"Corresponding TPR: {opt_tpr:.4f}")
    # print(f"Corresponding FPR: {opt_fpr:.4f}")

    return manual_auc, opt, thresholds, tpr_list, fpr_list

D = 1000
select = az.summary(result, var_names=["lambda_d"])['mean'].values
select = select[:-1]
truth = sim_dat["w"]

total_auc, total_youden, _, _, _ = calculate_auc(select,truth)
D_genotype = round(D * r / (r + 1))
genotype_auc, _, _, _,_ = calculate_auc(select[np.arange(D_genotype)], truth[np.arange(D_genotype)])
mrna_auc, _, thresholds, tpr_list, fpr_list = calculate_auc(select[np.arange(D_genotype,D)], truth[np.arange(D_genotype,D)])
print("AUC: ",total_auc, "genotype_AUC: ",genotype_auc,"mrna_AUC: ",mrna_auc,flush=True)

In [ ]:
# 3 get selection index by an empirical value
lambda_d_samples = result.posterior['lambda_d'].values

lambda_d_samples = lambda_d_samples[:, :, :-1]
lambda_d_samples = lambda_d_samples.reshape(4000, 1000)

lambda_map = Parallel(n_jobs=num_cores)(
    delayed(calculate_marginal_map)(lambda_d_samples[:, i]) for i in range(1000)
)

select = np.array(lambda_map)
# select = az.summary(result, var_names=["lambda_d"])['mean'].values
# select = select[:-1]
print(max(select))

truth = sim_dat["w"]
t = 0.52
y_pred = (select > t).astype(int)
TP = np.sum((y_pred == 1) & (truth == 1))
FP = np.sum((y_pred == 1) & (truth == 0))
FN = np.sum((y_pred == 0) & (truth == 1))
TN = np.sum((y_pred == 0) & (truth == 0))

sensitivity = TP / (TP + FN)
specificity = TN / (FP + TN)

print("Sensitivity: ", sensitivity, "Specificity: ", specificity)

In [ ]:
youden_j = np.array(tpr_list) - np.array(fpr_list)
opt_index = np.argmax(youden_j)
opt = thresholds[opt_index]
opt_tpr = tpr_list[opt_index]
opt_fpr = fpr_list[opt_index]

print(f"\nOptimal threshold (Youden's J): {opt:.4f}")
print(f"Corresponding TPR: {opt_tpr:.4f}")
print(f"Corresponding FPR: {opt_fpr:.4f}")

plt.figure()
plt.plot(fpr_list, tpr_list, color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')

plt.text(0.95, 0.05, f'AUC={total_auc:.2f}',
         horizontalalignment='right',
         verticalalignment='bottom',
         fontsize=10,
         color='black')

# plt.scatter(opt_fpr, opt_tpr, color='red', s=100, marker='o', label=f'Optimal (Youden\'s J): {opt:.2f}')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate',fontsize=10)
plt.ylabel('True Positive Rate',fontsize=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10) 
# plt.legend(loc="lower right",fontsize=18)
plt.grid(True)
# plt.savefig(f'Fig_MCMC/AUC_k{K}n{N}r{r}tr{tr}.pdf', format='pdf', bbox_inches='tight', transparent=True)

plt.show()

In [ ]:
eta_kd_samples = result.posterior['eta_kd'].values
eta_kd_samples = eta_kd_samples[:, :, :-1, :]

eta_kd_samples = eta_kd_samples.reshape(4000, 1000, K)

eta_kd_map = Parallel(n_jobs=num_cores)(
    delayed(calculate_marginal_map)(eta_kd_samples[:, i, j]) 
    for i in range(1000) 
    for j in range(K)
)

eta_kd_map = np.array(eta_kd_map).reshape((1000, K))

# hdi_prob = 50%
hdi_50 = az.summary(result, var_names=["eta_kd"], hdi_prob=0.5)
hdi_50_lr = hdi_50["hdi_25%"].values
hdi_50_lr = hdi_50_lr.reshape((1001, K))
hdi_50_lr = hdi_50_lr[:-1,:]
hdi_50_ur = hdi_50["hdi_75%"].values
hdi_50_ur = hdi_50_ur.reshape((1001, K))
hdi_50_ur = hdi_50_ur[:-1,:]


# hdi_prob = 90%
hdi_90 = az.summary(result, var_names=["eta_kd"], hdi_prob=0.9)
hdi_90_lr = hdi_90["hdi_5%"].values
hdi_90_lr = hdi_90_lr.reshape((1001, K))
hdi_90_lr = hdi_90_lr[:-1,:]
hdi_90_ur = hdi_90["hdi_95%"].values
hdi_90_ur = hdi_90_ur.reshape((1001, K))
hdi_90_ur = hdi_90_ur[:-1,:]


df = np.column_stack((select, truth, eta_kd_map, hdi_50_lr, hdi_50_ur, hdi_90_lr, hdi_90_ur, sim_dat["eta"].T))
column_names = ['select','truth','c1_map', 'c2_map', 'c3_map','c1_50_lr', 'c2_50_lr', 
                'c3_50_lr','c1_50_ur', 'c2_50_ur', 'c3_50_ur','c1_90_lr', 'c2_90_lr',
                'c3_90_lr','c1_90_ur', 'c2_90_ur', 'c3_90_ur','c1_true',"c2_true","c3_true"]
# column_names = ['select','truth','c1_map', 'c2_map', 'c3_map','c4_map', 'c5_map','c1_50_lr', 'c2_50_lr', 
#                 'c3_50_lr','c4_50_lr', 'c5_50_lr','c1_50_ur', 'c2_50_ur', 'c3_50_ur','c4_50_ur', 'c5_50_ur','c1_90_lr', 'c2_90_lr',
#                 'c3_90_lr','c4_90_lr', 'c5_90_lr','c1_90_ur', 'c2_90_ur', 'c3_90_ur','c4_90_ur', 'c5_90_ur',
#                 'c1_true',"c2_true","c3_true","c4_true","c5_true"]
header_line = "\t".join(column_names)
np.savetxt(f"File_MCMC/Lambda_k{K}n{N}r{r}tr{tr}.txt",df,delimiter="\t",fmt='%.6f',header=header_line,comments='')

In [ ]:
# order indices
eta_kd_true = sim_dat["eta"]
eta_kd_true = eta_kd_true[indice_true,:]

# hdi_prob = 95%
hdi_95 = az.summary(result, var_names=["eta_kd"], hdi_prob=0.95)
hdi_95_lr = hdi_95["hdi_2.5%"].values
D = 1000
hdi_95_lr = hdi_95_lr.reshape((D+1, K))
hdi_95_lr = hdi_95_lr[:-1,:]
hdi_95_lr = hdi_95_lr[:,indice_est]
hdi_95_lr = hdi_95_lr.T

hdi_95_ur = hdi_95["hdi_97.5%"].values
hdi_95_ur = hdi_95_ur.reshape((D+1, K))
hdi_95_ur = hdi_95_ur[:-1,:]
hdi_95_ur = hdi_95_ur[:,indice_est]
hdi_95_ur = hdi_95_ur.T

coverage_mask = (eta_kd_true >= hdi_95_lr) & (eta_kd_true <= hdi_95_ur)
coverage_count = np.sum(coverage_mask)